# PaddleOCR – Texterkennung mit Python

In diesem Notebook baust du Schritt für Schritt eine Texterkennung.

**Was passiert hier?**
1. Du lädst ein Bild
2. Ein KI-Modell erkennt den Text darin
3. Du zeichnest farbige Boxen um den erkannten Text

---


In [ ]:
%pip install matplotlib

## Setup (nicht ändern)

In [ ]:
import os
import time
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Konfiguration für PaddleOCR
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["FLAGS_allocator_strategy"] = "auto_growth"
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

from paddleocr import PaddleOCR

print("✅ Imports erfolgreich!")


## Modell laden (nicht ändern)

Hier wird das OCR-Modell konfiguriert und geladen.
Das kann beim ersten Mal ein paar Sekunden dauern.


In [ ]:
MODES = {
    "Multilingual (DE/EN/FR/CH/...)": {
        "lang": "german",
        "rec_mobile": "PP-OCRv5_mobile_rec",
        "rec_server": "PP-OCRv5_server_rec",
    },
    "Arabic": {
        "lang": "ar",
        "rec_mobile": "arabic_PP-OCRv5_mobile_rec",
        "rec_server": "arabic_PP-OCRv5_mobile_rec",
    },
}

def get_model(mode="Multilingual (DE/EN/FR/CH/...)", mobile=True):
    cfg = MODES[mode]
    rec_model = cfg["rec_mobile"] if mobile else cfg["rec_server"]
    kwargs = dict(
        lang=cfg["lang"],
        device="cpu",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        text_recognition_model_name=rec_model,
    )
    if mobile:
        kwargs["text_detection_model_name"] = "PP-OCRv5_mobile_det"
    return PaddleOCR(**kwargs)

model = get_model()
print("✅ Modell geladen!")


## TODO 1: Bild laden

Lade ein Bild aus dem Ordner `test_imgs/` und wandle es in ein numpy Array um.

**Tipps:**
- `Image.open("pfad/zum/bild.jpg")` öffnet eine Bilddatei
- `.convert("RGB")` stellt sicher dass es 3 Farbkanäle hat
- `np.array(...)` wandelt ein PIL-Bild in ein numpy Array um


In [ ]:
# TODO: Bild laden
# Beispiel: image = np.array(Image.open("test_imgs/general_ocr.png").convert("RGB"))

image = None  # <-- ersetze das

# Bild anzeigen (funktioniert sobald image kein None mehr ist)
if image is not None:
    plt.figure(figsize=(10, 8))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Bild geladen: {image.shape[1]}x{image.shape[0]} Pixel")
    plt.show()


## OCR ausführen (nicht ändern)

Dieser Teil schickt das Bild an das Modell und misst die Zeit.


In [ ]:
assert image is not None, "Bitte zuerst ein Bild laden (TODO 1)!"

start = time.perf_counter()
results = list(model.predict(image.copy()))
elapsed = time.perf_counter() - start

print(f"✅ Fertig in {elapsed*1000:.0f}ms")
print(f"   Anzahl Ergebnisse: {len(results)}")


## TODO 2: Detections aus Ergebnissen bauen

Die rohen Ergebnisse müssen in eine übersichtliche Liste umgewandelt werden.

Jede Detection soll so aussehen:
```python
{
    "text": "Hallo Welt",
    "confidence": 0.98,
    "polygon": [[10, 20], [100, 20], [100, 40], [10, 40]]
}
```

**Tipps:**
- `result.get("rec_polys", [])` gibt die Polygone zurück
- `result.get("rec_texts", [])` gibt die Texte zurück  
- `result.get("rec_scores", [])` gibt die Konfidenzwerte zurück
- `zip(liste1, liste2, liste3)` verbindet drei Listen elementweise:
  ```python
  for a, b, c in zip([1,2], ["x","y"], [True, False]):
      print(a, b, c)  # → 1 x True, dann 2 y False
  ```


In [ ]:
detections = []

for result in results:
    pass  # <-- ersetze pass mit deinem Code
    # Tipp: for poly, text, score in zip(..., ..., ...):
    #           detections.append({...})

print(f"Gefundene Texte: {len(detections)}")
if detections:
    print("Erstes Element:", detections[0])


## TODO 3: Farbe anhand Konfidenz berechnen

Schreibe eine Funktion die eine Konfidenz (0.0 bis 1.0) in eine Farbe umwandelt:
- `conf = 1.0` → grün `(0, 255, 0)`
- `conf = 0.0` → rot `(255, 0, 0)`
- `conf = 0.5` → gelb `(127, 127, 0)`

**Tipp:** `int(conf * 255)` ergibt einen Wert zwischen 0 und 255


In [ ]:
def conf_to_color(conf):
    # TODO: Berechne r und g anhand von conf
    r = 255  # <-- ersetze das
    g = 0    # <-- ersetze das
    return (r, g, 0)

# Test:
print("conf=1.0 →", conf_to_color(1.0), " (sollte (0, 255, 0) sein)")
print("conf=0.0 →", conf_to_color(0.0), " (sollte (255, 0, 0) sein)")
print("conf=0.5 →", conf_to_color(0.5), " (sollte (127, 127, 0) sein)")


## TODO 4: Boxen auf das Bild zeichnen

Zeichne für jede Detection ein farbiges Polygon auf das Bild.

**Tipps:**
- `ImageDraw.Draw(img)` erstellt ein Zeichen-Objekt
- `draw.polygon(poly, outline=color, width=2)` zeichnet ein Polygon
- `[tuple(p) for p in det["polygon"]]` wandelt die Punktliste um


In [ ]:
def build_image(image_rgb, detections):
    img = Image.fromarray(image_rgb)
    draw = ImageDraw.Draw(img)

    for det in detections:
        poly = [tuple(p) for p in det["polygon"]]
        conf = det.get("confidence", 0)

        color = conf_to_color(conf)

        # TODO: Zeichne das Polygon auf das Bild

    return np.array(img)

# Ergebnis anzeigen
result_img = build_image(image, detections)
plt.figure(figsize=(12, 10))
plt.imshow(result_img)
plt.axis("off")
plt.title(f"{len(detections)} Texte erkannt in {elapsed*1000:.0f}ms")
plt.show()


## Bonus: Erkannte Texte ausgeben

Gib alle erkannten Texte als Liste aus.  
Format: `"Hallo Welt" (95%)`


In [ ]:
# TODO Bonus: Texte ausgeben
for det in detections:
    pass  # <-- ersetze pass mit deinem Code
